# Paper 3 — 03 · RD-DPO training

Trains one (anchor, condition, seed) tuple per run. Conditions:

| Condition           | Layers updated                              | Description                  |
|---------------------|---------------------------------------------|------------------------------|
| `sft`               | All linear modules, all blocks (LoRA r=16)  | SFT on chosen only           |
| `rd-dpo-k1/2/4/8`   | LoRA r=16 on top-k probe-selected blocks    | RD-DPO with k targeted blocks|
| `dpo-lora-all`      | All linear modules, all blocks (LoRA r=16)  | Standard LoRA-DPO baseline   |
| `dpo-full`          | Every parameter                             | Vanilla full-model DPO       |

**A100-40G runtime** (one epoch, ~12k pairs, seq 1024, effective batch 32):

| Anchor       | dpo-full | dpo-lora-all | rd-dpo-k4 |
|--------------|----------|--------------|-----------|
| Qwen-2.5-3B  | ~3 h     | ~2 h         | ~1 h      |
| Llama-3.2-3B | ~3 h     | ~2 h         | ~1 h      |
| Gemma-3-4B   | ~4 h     | ~2.5 h       | ~1.3 h    |

In [ ]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    'torchao>=0.16' \
    huggingface_hub ipywidgets pyyaml -q


In [ ]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


# --- Paper 3 single-source-of-truth helpers ---
# short_of and family_of imported from src/prompts.py so the notebook
# convention stays aligned with notebook 02 and notebook 02b.
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that "
        "path in Drive, then re-run this cell."
    )
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    short_of, family_of,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    verify_smoke_gate, SmokeGateNotPassed,
)

print("Bootstrap done.")


## Pre-flight gate

Verifies smoke gate is fresh, the augmented preferences file
(`preferences_v2.jsonl`) exists, and the probe artefact
(`selected_blocks.json`) exists for the configured anchor.
Training will not start without all three.


In [ ]:
ANCHOR_PREFLIGHT = "Qwen/Qwen2.5-3B-Instruct"   # match cfg cell
short_pre  = short_of(ANCHOR_PREFLIGHT)

print("Pre-flight gate (PAPER3_PLAN section 15.7)")
print("-" * 72)
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR_PREFLIGHT)
except SmokeGateNotPassed as e:
    raise SystemExit(f"\nPRE-FLIGHT FAIL\n{e}\n")

pairs_path_pre = PREFS_DIR / short_pre / "preferences_v2.jsonl"
if not pairs_path_pre.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Augmented preferences not found at {pairs_path_pre}.\n"
        f"Run notebook 02b (Stage 4 augmentation) for {short_pre} first."
    )

probe_path_pre = PROBE_DIR / short_pre / "selected_blocks.json"
if not probe_path_pre.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Probe artefacts not found at {probe_path_pre}.\n"
        f"Run notebook 01 (refusal-direction probe) for {short_pre} first."
    )

print(f"smoke ok               : {smoke['completed_at']}")
print(f"preferences_v2 found   : {pairs_path_pre}")
print(f"selected_blocks found  : {probe_path_pre}")
print(f"\nPRE-FLIGHT PASS. Anchor {short_pre!r} is ready for training.")


## Configuration

`SEED` cycles `{17, 1729, 65537}` per `configs/models.yaml`.

In [ ]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4"
SEED      = 17

# Locked at week 1 (METHOD §4.2). Per-anchor tuning of these is forbidden.
BETA              = 0.1
LR                = 5e-6
EPOCHS            = 3        # small dataset -> need more epochs
WARMUP_STEPS      = 10       # was 100; never fired at 5-6 steps/epoch
MAX_STEPS         = 200      # cap so larger datasets do not run away
MAX_SEQ_LEN       = 1024
BATCH_PER_DEV     = 4
GRAD_ACCUM_STEPS  = 8       # effective batch 32
LORA_R, LORA_A    = 16, 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
RUN_DIR = ADAPTERS_DIR / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run    : {RUN_TAG}")
print(f"adapter: {RUN_DIR}")


## Resolve probe-selected blocks

For RD-DPO conditions only — `dpo-lora-all` / `dpo-full` / `sft` ignore.

In [ ]:
selected_blocks = None
if CONDITION.startswith("rd-dpo-"):
    k = int(CONDITION.split("-k")[-1])
    sel_path = PROBE_DIR / short / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(
            f"Probe artefact not found at {sel_path}. "
            f"Run 01_refusal_probe.ipynb first."
        )
    sel = json.loads(sel_path.read_text())
    selected_blocks = sel[str(k)]
    print(f"selected blocks (k={k}): {selected_blocks}")
else:
    print("(non-RD condition; LoRA targets all blocks or full model)")

## Load preferences + build dataset

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

pairs_path = PREFS_DIR / short / "preferences_v2.jsonl"
if not pairs_path.exists():
    raise FileNotFoundError(
        f"Preference dataset not found at {pairs_path}. "
        f"Run notebook 02 (Stage 1+2) and notebook 02b (Stage 4 augmentation) first."
    )
pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
print(f"loaded {len(pairs)} pairs from {pairs_path}")

tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token

def format_row(r):
    prompt_chat = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

ds_full   = Dataset.from_list([format_row(r) for r in pairs])
splits    = ds_full.train_test_split(test_size=0.05, seed=SEED)
ds_train  = splits["train"]; ds_eval = splits["test"]
print({"train": len(ds_train), "eval": len(ds_eval)})


## Build LoRA target spec

PEFT's `target_modules` accepts a list of module name suffixes or a regex. For
RD-DPO we restrict by **block index** via a regex matching `layers.{i}.…`.

In [ ]:
from peft import LoraConfig

if CONDITION == "dpo-full":
    lora_config = None
elif CONDITION in ("sft", "dpo-lora-all"):
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    )
elif CONDITION.startswith("rd-dpo-"):
    block_pat = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(LORA_TARGET_MODULES)
    target_re = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )
else:
    raise ValueError(f"unknown condition {CONDITION!r}")
print("lora_config:", lora_config)

## Load model + train

A100 footprint:
- bf16 throughout (Paper 2 R-Gemma-3 lesson: never fp16).
- Gradient checkpointing on.
- TRL precomputes reference log-probs on dev (avoids carrying the reference model forward through every step).
- For `dpo-full`, expect ~38 GB peak; for LoRA conditions, ~16-22 GB.

In [ ]:
from transformers import AutoModelForCausalLM, set_seed
from trl import DPOConfig, DPOTrainer

set_seed(SEED)

model = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

if CONDITION == "sft":
    from trl import SFTTrainer, SFTConfig
    sft_cfg = SFTConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=25, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        dataset_text_field="chosen", max_length=MAX_SEQ_LEN,
        max_steps=MAX_STEPS,
    )
    trainer = SFTTrainer(
        model=model, args=sft_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )
else:
    dpo_cfg = DPOConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=25, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        beta=BETA, loss_type="sigmoid",
        max_length=MAX_SEQ_LEN,
        precompute_ref_log_probs=True,
        max_steps=MAX_STEPS,
    )
    trainer = DPOTrainer(
        model=model, args=dpo_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )

print("starting training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"training done in {elapsed/60:.1f} min")


## Save adapter + run-meta

In [ ]:
trainer.save_model(str(RUN_DIR))
peak_mem = torch.cuda.max_memory_allocated() / 1e9

(RUN_DIR / "run_meta.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "anchor": ANCHOR, "condition": CONDITION, "seed": SEED,
    "selected_blocks": selected_blocks,
    "preference_dataset": str(pairs_path),
    "n_pairs": len(pairs),
    "training_compute": {
        "wallclock_seconds": round(elapsed, 1),
        "peak_gpu_memory_gb": round(peak_mem, 2),
        "device": DEVICE_NAME,
    },
    "hyperparams": {
        "beta": BETA, "lr": LR, "epochs": EPOCHS,
        "lora_r": LORA_R, "lora_alpha": LORA_A,
        "max_seq_len": MAX_SEQ_LEN,
        "per_device_train_batch_size": BATCH_PER_DEV,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    },
    "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))
print(f"saved adapter + meta -> {RUN_DIR}")